In [1]:
import numpy as np

def evaluate_iecc_viability(
    category: str,
    first_cost: float,
    annual_energy_savings: float,
    maintenance_schedule: dict
):
    """
    Evaluates a proposal against IECC 2030 Cost Effectiveness criteria.

    Args:
        category: "Residential", "Multifamily", "Commercial", or "Industrial".
        first_cost: Initial implementation cost ($).
        annual_energy_savings: Projected annual savings ($).
        maintenance_schedule: Dict mapping year (int) to maintenance cost ($).
                              e.g., {10: 500} means $500 cost in year 10.
    """

    # --- 1. Define Constraints (Per Page 8 Table) ---
    PAYBACK_CAPS = {
        "Residential": 7,
        "Multifamily": 5,
        "Commercial": 5,
        "Industrial": 10
    }

    if category not in PAYBACK_CAPS:
        raise ValueError(f"Unknown category: {category}")

    max_payback_years = PAYBACK_CAPS[category]

    # --- 2. IECC [A] Test: Simple Payback ---
    # Logic: Payback = Net Cost / Annual Savings
    # Note: Commercial/Industrial logic often sums maintenance into cost for
    # simple payback metrics, whereas Residential is often pure First Cost.
    # Based on PDF logic, we treat maintenance as an additive cost impact.

    total_maintenance_nominal = sum(maintenance_schedule.values())

    if category == "Residential":
        # Residential usually looks at simple first cost recovery
        numerator = first_cost
    else:
        # Commercial includes lifecycle maintenance impact in the numerator
        # or treats it as a reduction in savings. Conservative approach:
        numerator = first_cost + total_maintenance_nominal

    simple_payback = numerator / annual_energy_savings if annual_energy_savings > 0 else float('inf')

    print(f"--- Analysis for {category} Proposal ---")
    print(f"Simple Payback: {simple_payback:.2f} years (Cap: {max_payback_years})")

    if simple_payback <= max_payback_years:
        return "✅ PASSED: Meets IECC Core [A] Criteria (Simple Payback)"

    # --- 3. IECC [B] Test: Net Present Value (NPV) ---
    # Required Parameters from PDF:
    DISCOUNT_RATE = 0.045  # 4.5%
    LIFECYCLE_YEARS = 30

    npv = -first_cost # Initial outflow

    for year in range(1, LIFECYCLE_YEARS + 1):
        # Inflow = Energy Savings - Maintenance Cost for that specific year
        maint_cost = maintenance_schedule.get(year, 0)
        cash_flow = annual_energy_savings - maint_cost

        # Discounted cash flow
        present_value = cash_flow / ((1 + DISCOUNT_RATE) ** year)
        npv += present_value

    print(f"NPV (30yr @ 4.5%): ${npv:,.2f}")

    if npv > 0:
        return "✅ PASSED: Meets IECC Advanced [B] Criteria (Positive NPV)"

    return "❌ FAILED: Does not meet [A] or [B] criteria."

# --- Run Simulations ---
if __name__ == "__main__":
    # Case 1: A cheap upgrade with quick return (Should pass [A])
    result_1 = evaluate_iecc_viability(
        category="Residential",
        first_cost=1000,
        annual_energy_savings=200,
        maintenance_schedule={}
    )
    print(f"Result: {result_1}\n")

    # Case 2: Expensive Commercial system, high maintenance, but massive long-term savings
    # Fails Simple Payback (Cap 5), Checks NPV
    result_2 = evaluate_iecc_viability(
        category="Commercial",
        first_cost=50000,
        annual_energy_savings=6000,
        maintenance_schedule={10: 5000, 20: 5000} # Maintenance every 10 years
    )
    print(f"Result: {result_2}\n")

    # Case 3: Inefficient proposal
    result_3 = evaluate_iecc_viability(
        category="Multifamily",
        first_cost=10000,
        annual_energy_savings=500,
        maintenance_schedule={}
    )
    print(f"Result: {result_3}\n")

--- Analysis for Residential Proposal ---
Simple Payback: 5.00 years (Cap: 7)
Result: ✅ PASSED: Meets IECC Core [A] Criteria (Simple Payback)

--- Analysis for Commercial Proposal ---
Simple Payback: 10.00 years (Cap: 5)
NPV (30yr @ 4.5%): $42,440.48
Result: ✅ PASSED: Meets IECC Advanced [B] Criteria (Positive NPV)

--- Analysis for Multifamily Proposal ---
Simple Payback: 20.00 years (Cap: 5)
NPV (30yr @ 4.5%): $-1,855.56
Result: ❌ FAILED: Does not meet [A] or [B] criteria.

